In [ ]:
# 필수 라이브러리 임포트
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("서울시 100m 격자 인구 데이터 통합 스크립트")
print("=" * 80)
print("\n라이브러리 로드 완료")

In [ ]:
# 경로 설정
# 상대 경로 사용 (현재 파일 기준)
base_path = Path(__file__).parent.parent / 's1_data' / 's1_org_data' if '__file__' in dir() else Path('../s1_data/s1_org_data')
output_path = Path(__file__).parent.parent / 's1_data' / 's2_processed_data' if '__file__' in dir() else Path('../s1_data/s2_processed_data')
output_path.mkdir(parents=True, exist_ok=True)
grid_path = base_path / '국토정보플랫폼' / '시군구격자인구수'

print(f"\n격자 데이터 경로: {grid_path}")
print(f"출력 경로: {output_path}")
print(f"경로 존재 확인: {grid_path.exists()}")

In [ ]:
print("\n" + "=" * 80)
print("시군구별 격자 폴더 탐색")
print("=" * 80)

sigungu_folders = [d for d in grid_path.iterdir() if d.is_dir()]

print(f"\n발견된 시군구 폴더 수: {len(sigungu_folders)}개")
print("\n시군구 목록:")
for i, folder in enumerate(sigungu_folders, 1):
    print(f"  {i:2d}. {folder.name}")

In [ ]:
print("\n" + "=" * 80)
print("SHP 파일 탐색")
print("=" * 80)

shp_files = []

for folder in sigungu_folders:
    shp_list = list(folder.glob('*.shp'))

    if shp_list:
        shp_files.extend(shp_list)
        print(f"✓ {folder.name.split('_')[-2]}: {len(shp_list)}개 SHP 파일")
    else:
        print(f"✗ {folder.name.split('_')[-2]}: SHP 파일 없음")

print(f"\n총 발견된 SHP 파일 수: {len(shp_files)}개")

In [ ]:
print("\n" + "=" * 80)
print("샘플 파일 구조 확인")
print("=" * 80)

if shp_files:
    sample_gdf = gpd.read_file(shp_files[0], encoding='utf-8')

    print(f"\n파일: {shp_files[0].name}")
    print(f"데이터 행 수: {len(sample_gdf)}")
    print(f"컬럼: {list(sample_gdf.columns)}")
    print(f"좌표계: {sample_gdf.crs}")
    print(f"\n데이터 타입:")
    print(sample_gdf.dtypes)
    print(f"\n상위 5개 데이터:")
    print(sample_gdf.head())
else:
    print("SHP 파일을 찾을 수 없습니다!")
    exit(1)

In [ ]:
print("\n" + "=" * 80)
print("SHP 파일 통합 시작")
print("=" * 80)

all_gdfs = []

for i, shp_file in enumerate(shp_files, 1):
    try:
        gdf = gpd.read_file(shp_file, encoding='utf-8')

        # 시군구명 추출
        sigungu_name = shp_file.parent.name.split('_')[-2]
        gdf['SIGUNGU'] = sigungu_name

        all_gdfs.append(gdf)

        print(f"  [{i:2d}/{len(shp_files)}] {sigungu_name}: {len(gdf):,}개 격자 로드")

    except Exception as e:
        print(f"  ✗ [{i:2d}/{len(shp_files)}] {shp_file.name} 로드 실패: {e}")

print(f"\n✓ 총 {len(all_gdfs)}개 파일 로드 완료")

# 모든 데이터프레임 합치기
print("\n데이터 통합 중...")
gdf_combined = gpd.GeoDataFrame(pd.concat(all_gdfs, ignore_index=True))

print(f"\n✓ 통합 완료!")
print(f"  - 총 격자 수 (통합 전): {len(gdf_combined):,}개")
print(f"  - 컬럼: {list(gdf_combined.columns)}")
print(f"  - 좌표계: {gdf_combined.crs}")

In [ ]:
print("\n" + "=" * 80)
print("통합 전 데이터 확인")
print("=" * 80)

print(f"\n총 행 수: {len(gdf_combined):,}")
print(f"\n컬럼별 결측치:")
print(gdf_combined.isnull().sum())
print(f"\nGID 고유값 수: {gdf_combined['gid'].nunique():,}개")
print(f"GID 중복 수: {len(gdf_combined) - gdf_combined['gid'].nunique():,}개")
print(f"\nVAL(인구수) 통계:")
print(gdf_combined['val'].describe())
print(f"\n총 인구수: {gdf_combined['val'].sum():,}명")

In [ ]:
gdf_combined['val'] = gdf_combined['val'].fillna(0.0)
gdf_combined = gdf_combined[['gid', 'val', 'geometry']]


In [ ]:
gdf_combined

In [ ]:
print("\n" + "=" * 80)
print("GID 그룹화 및 VAL 합산")
print("=" * 80)

gdf_merged = gdf_combined.groupby('gid').agg({
    'geometry': 'first',
    'val': 'sum',
}).reset_index()

gdf_merged = gpd.GeoDataFrame(gdf_merged, geometry='geometry', crs=gdf_combined.crs)

print(f"\n✓ 그룹화 완료!")
print(f"  - 그룹화 전: {len(gdf_combined):,}개")
print(f"  - 그룹화 후: {len(gdf_merged):,}개")
print(f"  - 중복 제거: {len(gdf_combined) - len(gdf_merged):,}개")

# 좌표계 확인 및 변환
print(f"\n현재 좌표계: {gdf_merged.crs}")

if gdf_merged.crs != 'EPSG:5179':
    print("EPSG:5179로 변환 중...")
    gdf_merged = gdf_merged.to_crs('EPSG:5179')
    print(f"✓ 변환 완료: {gdf_merged.crs}")
else:
    print("✓ 이미 EPSG:5179 좌표계입니다.")

In [ ]:
# ============================================================================
# 10. 결과 저장
# ============================================================================
print("\n" + "=" * 80)
print("결과 저장")
print("=" * 80)

output_shp = output_path / 'seoul_100m_grid_population.shp'
gdf_merged.to_file(output_shp, encoding='utf-8')
print(f"✓ Shapefile 저장 완료: {output_shp}")

output_geojson = output_path / 'seoul_100m_grid_population.geojson'
gdf_merged.to_file(output_geojson, driver='GeoJSON', encoding='utf-8')
print(f"✓ GeoJSON 저장 완료: {output_geojson}")

output_csv = output_path / 'seoul_100m_grid_population.csv'
gdf_merged.drop(columns=['geometry']).to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"✓ CSV 저장 완료: {output_csv}")